# Regulatory Document Processing Pipeline with Textract

Universal document processing pipeline using AWS Textract and Bedrock for any file format.

## 🆕 Recent Updates: Intelligent Chunking for Large Documents

**What's New**: The extraction pipeline now handles large documents using intelligent chunking that ONLY activates when the model actually can't handle the input size!

### Key Improvements:
1. **Full Document Processing**: Processes entire documents up to 200K tokens (Claude 4.5)
2. **Smart Chunking**: Only splits documents when the model explicitly says the input is too long
3. **No False Positives**: Won't split for authentication errors, model ID errors, or other API issues
4. **Automatic Merging**: Results from multiple chunks are intelligently merged when splitting occurs
5. **Better Error Handling**: Clear error messages distinguish between length issues and other problems

### How It Works:
```
1. Try to process the ENTIRE document first
2. If model says "input too long" → split in half
3. Process each half (recursively if needed)
4. Merge results intelligently
5. For any other error → report it, don't split
```

### When Splitting Happens:
The system ONLY splits when it detects these specific errors:
- ✅ "prompt is too long"
- ✅ "too many tokens"
- ✅ "maximum context length"
- ✅ "input is too long"

It will NOT split for:
- ❌ Invalid model ID errors
- ❌ Authentication errors
- ❌ Rate limiting (handles with backoff instead)
- ❌ Invalid request format
- ❌ Any other API errors

### Merging Strategy (when splitting occurs):
- **Title**: Longest/most complete version
- **Country/Date**: First non-empty value
- **Sectors/Requirements**: Combined lists (deduplicated)
- **Penalties**: Most detailed description

### Model Used:
- **Claude 4.5 Sonnet ** (`anthropic.claude-sonnet-4-5-20250929-v1:0`)
- Supports up to 200K tokens input (~600K characters)
- Most documents will fit without splitting!

## 1. Dependencies & Imports

Import necessary libraries for AWS integration (Boto3), NLP processing (Transformers, PyTorch), document parsing (BeautifulSoup), and data handling.

In [ ]:
import boto3
import json
import time
import os
from typing import Dict, Any, Optional
from botocore.exceptions import ClientError
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModel, pipeline
import torch

In [ ]:
# Quick AWS credentials health check (optional to run)
import boto3, os
from botocore.exceptions import NoCredentialsError, ClientError

def aws_credentials_health_check(region: str = os.getenv('AWS_REGION') or os.getenv('AWS_DEFAULT_REGION') or 'us-east-1'):
    print(f"🔎 Checking AWS credentials (region={region})...")
    print(f"   AWS_ACCESS_KEY_ID set: {'YES' if bool(os.getenv('AWS_ACCESS_KEY_ID')) else 'NO'}")
    print(f"   AWS_SESSION_TOKEN set: {'YES' if bool(os.getenv('AWS_SESSION_TOKEN')) else 'NO'}")
    
    try:
        sts = boto3.client('sts', region_name=region)
        ident = sts.get_caller_identity()
        print(f"✅ Auth OK: Account={ident.get('Account')} Arn={ident.get('Arn')}")
        return True
    except NoCredentialsError:
        print("❌ NoCredentialsError: No AWS credentials found. Load a .env, configure ~/.aws, or export env vars.")
        return False
    except ClientError as e:
        code = e.response.get('Error', {}).get('Code', 'Unknown')
        print(f"❌ AWS ClientError: {code}")
        if code in ('ExpiredToken', 'ExpiredTokenException'):
            print("➡️ Your temporary session has expired. Refresh via AWS SSO or new STS tokens, then rerun.")
        return False

# Uncomment to run the check right away
# aws_credentials_health_check()

## 2. RegulatoryDocumentProcessor Class

Main processing class that orchestrates:
- **Text Extraction**: Handles multiple formats (HTML, XML, PDF, images) using AWS Textract and BeautifulSoup
- **Classification**: Uses LegalBERT to classify regulations into categories (Environmental, Financial, Trade, Privacy, Labor, Tax)
- **Key Information Extraction**: Leverages Claude via Bedrock API to extract critical compliance details

In [ ]:
class RegulatoryDocumentProcessor:
    def __init__(self):
        self.bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
        self.textract = boto3.client('textract', region_name='us-east-1')
        
        # Configure Bedrock model fallback strategy
        # Primary from env or default to Claude 4.5 Sonnet (may require an inference profile on some accounts)
        self.model_id_primary = os.getenv('BEDROCK_MODEL_ID', 'anthropic.claude-sonnet-4-5-20250929-v1:0')
        # Optional inference profile ARN for models that require profiles (e.g., Sonnet 4.5 in some accounts)
        self.inference_profile_arn = os.getenv('BEDROCK_INFERENCE_PROFILE_ARN')
        # Safe fallback widely available on-demand
        self.model_id_fallback = 'us.anthropic.claude-3-5-sonnet-20241022-v2:0'
        
        # Load LegalBERT for legal document classification
        print("📚 Loading LegalBERT model for legal analysis...")
        self.legal_tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.legal_model = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.legal_model.to(self.device)
        print("✅ LegalBERT model loaded successfully")
        
    def _invoke_bedrock(self, prompt: str, max_tokens: int = 1000):
        """Invoke Bedrock with fallback for inference-profile-only models."""
        payload = {
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': max_tokens,
            'messages': [{'role': 'user', 'content': prompt}]
        }
        # Try primary model id first
        try:
            return self.bedrock.invoke_model(modelId=self.model_id_primary, body=json.dumps(payload))
        except Exception as e:
            msg = str(e)
            # If the account requires an inference profile for the selected model, retry with profile or fallback
            if ('ValidationException' in msg) and ('inference profile' in msg.lower()):
                if self.inference_profile_arn:
                    try:
                        return self.bedrock.invoke_model(modelId=self.inference_profile_arn, body=json.dumps(payload))
                    except Exception as e2:
                        # If profile also fails, and primary wasn't the safe fallback, try safe fallback
                        if self.model_id_primary != self.model_id_fallback:
                            return self.bedrock.invoke_model(modelId=self.model_id_fallback, body=json.dumps(payload))
                        raise e2
                # No profile set: try safe fallback if not already used
                if self.model_id_primary != self.model_id_fallback:
                    return self.bedrock.invoke_model(modelId=self.model_id_fallback, body=json.dumps(payload))
            # Propagate any other errors
            raise
        
    def extract_text_from_file(self, file_path: str) -> str:
        """Extract text from any local file using Textract or BeautifulSoup"""
        print(f"📄 Extracting text from: {file_path.split('/')[-1]}")
        
        _, ext = os.path.splitext(file_path.lower())
        print(f"   File extension detected: {ext}")
        
        if ext in ['.html', '.xml']:
            # Use BeautifulSoup for HTML/XML FIRST (before Textract)
            print("   Using BeautifulSoup for HTML/XML parsing...")
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    soup = BeautifulSoup(file, 'html.parser')
                    text = soup.get_text(separator='\n')
                print(f"✅ Extracted {len(text):,} characters using BeautifulSoup")
                return text
            except Exception as e:
                print(f"❌ Error parsing HTML/XML: {e}")
                return ""
        
        elif ext in ['.pdf', '.jpg', '.jpeg', '.png', '.tiff', '.tif']:
            # Use Textract for images and PDFs
            print("   Using Textract for document extraction...")
            try:
                with open(file_path, 'rb') as file:
                    document_bytes = file.read()
                
                response = self.textract.detect_document_text(
                    Document={'Bytes': document_bytes}
                )
                
                text = ""
                for block in response['Blocks']:
                    if block['BlockType'] == 'LINE':
                        text += block['Text'] + "\n"
                
                print(f"✅ Extracted {len(text):,} characters using Textract")
                return text
            except ClientError as e:
                if e.response['Error']['Code'] == 'UnsupportedDocumentException':
                    print("❌ Unsupported document format. Textract supports JPEG, PNG, PDF, and TIFF files.")
                    return ""
                else:
                    raise
        
        else:
            print(f"❌ Unsupported file format: {ext}. Supported: PDF, images (JPEG/PNG/TIFF), HTML, XML.")
            return ""
    
    def classify_regulation_with_legalbert(self, text: str) -> Dict[str, Any]:
        """Classify regulation using LegalBERT for legal document understanding"""
        print("🔍 Classifying document using LegalBERT...")
        
        # Use first 10000 chars for classification (better context than just 2000)
        truncated_text = text[:10000]
        
        try:
            # Tokenize input for LegalBERT (limited to 512 tokens)
            inputs = self.legal_tokenizer(
                truncated_text[:2000],  # LegalBERT still needs small input
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)
            
            # Get embeddings from LegalBERT
            with torch.no_grad():
                outputs = self.legal_model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1)
            
            # Use embedding norm as confidence measure
            confidence = min(torch.norm(embeddings).item() / 10, 1.0)
            print(f"📋 LegalBERT confidence score: {confidence:.2%}")
            
            # Use Claude Sonnet for contextual classification (more text for better accuracy)
            prompt = f"""Based on this regulatory document excerpt, classify it into ONE category:
            - Environmental 
            - Financial
            - Trade
            - Privacy
            - Labor
            - Tax
            
            Document excerpt: {truncated_text}
            
            Return only the category name and a brief 1-2 sentence explanation."""
            
            response = self._invoke_bedrock(prompt, max_tokens=200)
            
            result = json.loads(response['body'].read())
            classification = result['content'][0]['text'].strip()
            print(f"📋 Document classification: {classification}")
            
            return {
                'category': classification,
                'legalbert_confidence': confidence
            }
        
        except Exception as e:
            msg = str(e)
            if ('ExpiredToken' in msg) or ('ExpiredTokenException' in msg):
                print("⚠️ Classification failed due to expired AWS session token.")
                return {'category': 'Unknown', 'legalbert_confidence': 0.0, 'error': 'ExpiredTokenException'}
            if ('ValidationException' in msg) and ('inference profile' in msg.lower()):
                print("⚠️ Classification failed: model requires an inference profile or unsupported on-demand.")
                return {'category': 'Unknown', 'legalbert_confidence': 0.0, 'error': 'ValidationException:InferenceProfileRequired'}
            print(f"⚠️ Classification error: {e}. Falling back to basic extraction.")
            return {'category': 'Unknown', 'legalbert_confidence': 0.0}
    
    def extract_key_info(self, text: str, category: str) -> Dict[str, Any]:
        """Extract key regulatory information using Claude with automatic chunking for large documents"""
        print(f"🔎 Extracting key information for {category} regulation...")
        
        # Claude Sonnet 4.5 supports up to ~3.5M chars
        MAX_CHARS_SONNET = 3500000
        text_length = len(text)
        
        # Truncate to Claude's limit if needed
        if text_length > MAX_CHARS_SONNET:
            text = text[:MAX_CHARS_SONNET]
            print(f"   ⚠️ Document very long ({text_length:,} → 3.5M chars)")
        
        prompt_template = """You are a financial regulatory intelligence analyst. Extract an exhaustive and precise structured summary from this {category} regulation for downstream financial compliance analytics.

        Return STRICT JSON (UTF-8, no markdown) in English with the following schema. Omit a field only if the regulation provides no signal at all (use null instead of guessing):
        {{
          \"title\": string,
          \"country\": string,
          \"effective_date\": string,
          \"regulatory_body\": string or null,
          \"scope\": string or null,
          \"affected_sectors\": [string],
          \"affected_entities\": [string],
          \"key_requirements\": [string],
          \"key_requirement_details\": [
            {{
              \"requirement\": string,
              \"article_reference\": string,
              \"metrics\": [
                {{
                  \"type\": string,
                  \"value\": number,
                  \"unit\": string,
                  \"description\": string
                }}
              ],
              \"deadline\": string or null,
              \"compliance_action\": string or null
            }}
          ],
          \"reporting_requirements\": [
            {{
              \"description\": string,
              \"frequency\": string or null,
              \"format\": string or null,
              \"article_reference\": string
            }}
          ],
          \"monetary_thresholds\": [
            {{
              \"amount\": number,
              \"currency\": string,
              \"description\": string,
              \"article_reference\": string
            }}
          ],
          \"quantitative_limits\": [
            {{
              \"value\": number,
              \"unit\": string,
              \"description\": string,
              \"article_reference\": string
            }}
          ],
          \"penalties\": [
            {{
              \"type\": string,
              \"amount\": number or null,
              \"currency\": string or null,
              \"imprisonment\": string or null,
              \"article_reference\": string
            }}
          ],
          \"enforcement_agencies\": [string],
          \"implementation_timeline\": {{
            \"publication_date\": string or null,
            \"transposition_deadline\": string or null,
            \"phased_milestones\": [string]
          }},
          \"notes\": [string],
          \"source_articles\": [string]
        }}

        Document excerpt: {document_text}

        Extraction guidelines:
        - Capture EVERY actionable obligation, threshold, reporting duty, or quantitative limit explicitly. Do not collapse multiple items into one summary.
        - Provide explicit article/paragraph references for every obligation, metric, or penalty whenever stated.
        - Preserve all numeric values exactly; convert spelled-out numbers to digits and include currency ISO codes (e.g., \"EUR\") or units (%, tonnes, MW).
        - Represent percentages as decimals (e.g., 12.5% → 0.125) and monetary amounts as numbers (no thousands separators).
        - Translate narrative text to English while keeping proper nouns in their original form.
        - If a field is unknown, use null instead of fabricating content.
        - Keep JSON valid and exhaustive—no markdown, comments, or trailing commas."""
        
        def merge_regulatory_info(info1: Dict, info2: Dict) -> Dict:
            """Merge two regulatory information dictionaries"""
            merged = info1.copy() if info1 else {}
            
            if not info2:
                return merged
            
            # Title: take the longer one
            if info2.get('title') and (not merged.get('title') or len(info2['title']) > len(merged.get('title', ''))):
                merged['title'] = info2['title']
            
            # Country: take first non-empty
            if info2.get('country') and not merged.get('country'):
                merged['country'] = info2['country']
            
            # Effective date: take first non-empty
            if info2.get('effective_date') and not merged.get('effective_date'):
                merged['effective_date'] = info2['effective_date']
            
            # Affected sectors: merge lists
            sectors1 = merged.get('affected_sectors', [])
            if isinstance(sectors1, str):
                sectors1 = [sectors1] if sectors1 else []
            sectors2 = info2.get('affected_sectors', [])
            if isinstance(sectors2, str):
                sectors2 = [sectors2] if sectors2 else []
            merged['affected_sectors'] = list(set(sectors1 + sectors2))
            
            # Key requirements: merge lists
            reqs1 = merged.get('key_requirements', [])
            if isinstance(reqs1, str):
                reqs1 = [reqs1] if reqs1 else []
            reqs2 = info2.get('key_requirements', [])
            if isinstance(reqs2, str):
                reqs2 = [reqs2] if reqs2 else []
            merged['key_requirements'] = list(set(reqs1 + reqs2))
            
            # Penalties: take the longer one
            if info2.get('penalties') and (not merged.get('penalties') or len(str(info2['penalties'])) > len(str(merged.get('penalties', '')))):
                merged['penalties'] = info2['penalties']
            
            return merged
        
        def extract_recursive(text_chunk: str, depth: int = 0, max_depth: int = 10) -> Dict:
            """Recursively extract information, splitting if document is too long"""
            if depth > max_depth:
                print(f"   {'  ' * depth}❌ Max depth ({max_depth}) reached")
                return {}
            
            chunk_length = len(text_chunk)
            
            # Stop splitting if chunk is too small (< 100 chars) - likely an API error, not length issue
            if chunk_length < 100:
                print(f"   {'  ' * depth}⚠️ Chunk too small ({chunk_length} chars), stopping recursion")
                return {}
            
            prompt = prompt_template.format(category=category, document_text=text_chunk)
            
            try:
                response = self._invoke_bedrock(prompt, max_tokens=1500)
                
                result = json.loads(response['body'].read())
                content = result['content'][0]['text']
                
                try:
                    extracted_info = json.loads(content)
                    print(f"   {'  ' * depth}✅ Chunk extracted ({chunk_length:,} chars)")
                    return extracted_info
                except json.JSONDecodeError:
                    print(f"   {'  ' * depth}⚠️ Invalid JSON, returning raw content")
                    return {'raw_content': content}
                    
            except Exception as e:
                error_msg = str(e).lower()
                
                # Early return on auth errors
                if ("expiredtoken" in error_msg) or ("expired token" in error_msg):
                    print("   ❌ AWS token expired during extraction.")
                    return {'error': 'ExpiredTokenException', 'message': 'AWS STS session expired.'}
                
                # ONLY split if it's specifically about the prompt/input being too long
                is_length_error = (
                    "prompt is too long" in error_msg or
                    "too many tokens" in error_msg or
                    "maximum context length" in error_msg or
                    "input is too long" in error_msg or
                    ("too long" in error_msg and "prompt" in error_msg)
                )
                
                # Handle inference-profile validation error explicitly
                if ("validationexception" in error_msg) and ("inference profile" in error_msg):
                    print("   ❌ Model requires an inference profile or unsupported on-demand.")
                    return {'error': 'ValidationException:InferenceProfileRequired', 'message': 'Model requires inference profile or unsupported on-demand.'}
                
                if is_length_error and chunk_length > 1000:
                    # Document is too long for the model, split it
                    print(f"   {'  ' * depth}⚠️ Input too long ({chunk_length:,} chars) → Splitting")
                    
                    mid_point = len(text_chunk) // 2
                    split_point = text_chunk.rfind('\n', mid_point - 10000, mid_point + 10000)
                    if split_point == -1:
                        split_point = mid_point
                    
                    chunk1 = text_chunk[:split_point]
                    chunk2 = text_chunk[split_point:]
                    
                    info1 = extract_recursive(chunk1, depth + 1, max_depth)
                    info2 = extract_recursive(chunk2, depth + 1, max_depth)
                    
                    if info1 and info2:
                        return merge_regulatory_info(info1, info2)
                    return info1 or info2
                    
                elif "throttling" in error_msg or "throttled" in error_msg:
                    import time
                    wait = min(30, (depth + 1) * 5)
                    print(f"   {'  ' * depth}⏳ Throttling, waiting {wait}s...")
                    time.sleep(wait)
                    return extract_recursive(text_chunk, depth, max_depth)
                    
                else:
                    # Any other error - don't try to split, just report it
                    print(f"   {'  ' * depth}❌ Error: {str(e)[:150]}")
                    return {}
        
        # Start recursive extraction
        extracted_info = extract_recursive(text)
        
        if extracted_info:
            if isinstance(extracted_info, dict) and (extracted_info.get('error') in ['ExpiredTokenException', 'ValidationException:InferenceProfileRequired']):
                print("⚠️ Extraction aborted due to AWS invocation error.")
                return extracted_info
            print("✅ Key information extracted successfully")
            return extracted_info
        else:
            print("⚠️ Extraction failed, returning basic structure")
            return {'raw_content': text[:1000] + '...'}
    
    def process_document(self, file_path: str) -> Optional[Dict[str, Any]]:
        """Process any document format"""
        print(f"🚀 Starting document processing pipeline...")
        text = self.extract_text_from_file(file_path)
        
        if not text:
            print("❌ Failed to extract text from document")
            return None
        
        classification = self.classify_regulation_with_legalbert(text)
        key_info = self.extract_key_info(text, classification.get('category', 'Unknown'))
        print("🎉 Processing completed successfully!")
        
        # Mark failed if we hit auth/invocation errors
        processing_status = 'completed'
        error_details = None
        err_tag = None
        if isinstance(key_info, dict) and key_info.get('error') in ['ExpiredTokenException', 'ValidationException:InferenceProfileRequired']:
            processing_status = 'failed'
            err_tag = key_info.get('error')
            error_details = key_info.get('message') or 'Bedrock invocation error'
        
        return {
            'document_id': file_path.split('/')[-1],
            'category': classification.get('category', 'Unknown'),
            'legalbert_confidence': classification.get('legalbert_confidence', 0.0),
            'extracted_info': key_info,
            'processing_status': processing_status,
            **({'error': f"{err_tag}: {error_details}"} if err_tag else {})
        }

## Process Any Document Format

## 3. Execute Full Processing Pipeline

Run the complete document processing workflow on a sample regulatory file. The pipeline will:
1. Locate the document file
2. Extract text based on file format
3. Classify the regulation type
4. Extract key compliance information
5. Return structured results

In [ ]:
# Update the document_path to point to a valid local file and add a simple fallback search.
# The pipeline now supports HTML/XML alongside PDF and image formats.
# Adjust the candidate paths list if your files live elsewhere

from pathlib import Path

candidate_paths = [
    os.path.join('../../data/raw/directives', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),
    # os.path.join('directive', '1.DIRECTIVE (UE) 20192161 DU PARLEMENT EUROPÉEN ET DU CONSEIL.html'),

]

document_path = None
for path in candidate_paths:
    if os.path.exists(path):
        document_path = path
        break

if not document_path:
    raise FileNotFoundError(
        "Unable to locate the sample document. Update candidate_paths with a valid file location."
    )

print(f"Using document: {document_path}")

processor = RegulatoryDocumentProcessor()
try:
    result = processor.process_document(document_path)
except FileNotFoundError:
    print(f"Error: File not found at {document_path}. Please check the path and ensure the file exists.")
    result = None

if result:
    print("\nProcessing Results:")
    print(f"Document: {result['document_id']}")
    print(f"Category: {result['category']}")
    print(f"Status: {result['processing_status']}")
    print("\nExtracted Information:")
    print(json.dumps(result['extracted_info'], indent=2))
    
    # Save single-file result to JSON as well
    output_dir = Path('../../data/generated/extracted_directives')
    output_dir.mkdir(parents=True, exist_ok=True)
    file_stem = Path(document_path).stem
    output_file = output_dir / f"{file_stem}_extracted.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Saved result to: {output_file}")
else:
    print("⚠️ No result produced.")

In [ ]:
# Process all files in the directives directory and export to JSON
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
env_path = Path.cwd()
if 'notebooks' in str(env_path):
    env_path = env_path.parent
env_file = env_path / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print("✅ AWS Credentials loaded from .env")
else:
    print("⚠️ .env file not found. Using system environment variables.")

# Set output directory for regulatory files
output_dir = Path('../../data/generated/extracted_directives')
output_dir.mkdir(parents=True, exist_ok=True)
print(f"📁 Output directory: {output_dir}")

directives_dir = Path('../../data/raw/directives')

if directives_dir.exists():
    # Get all HTML/XML files in the directives directory
    all_files = list(directives_dir.glob('*.html')) + list(directives_dir.glob('*.xml'))
    
    if all_files:
        print(f"📋 Found {len(all_files)} regulatory files to process")
        print("=" * 80)
        
        results = []
        
        for idx, file_path in enumerate(all_files, 1):
            print(f"\n🔄 Processing {idx}/{len(all_files)}: {file_path.name}")
            print("-" * 80)
            
            try:
                processor = RegulatoryDocumentProcessor()
                result = processor.process_document(str(file_path))
                
                if result:
                    results.append(result)
                    
                    # 💾 Save individual JSON file
                    file_stem = file_path.stem
                    output_file = output_dir / f"{file_stem}_extracted.json"
                    with open(output_file, 'w', encoding='utf-8') as f:
                        json.dump(result, f, indent=2, ensure_ascii=False)
                    print(f"✅ Successfully processed and saved: {file_path.name}")
                    print(f"   📄 Saved to: {output_file}")
                else:
                    print(f"⚠️ Failed to process: {file_path.name}")
            
            except Exception as e:
                print(f"❌ Error processing {file_path.name}: {str(e)[:200]}")
                continue
        
        print("\n" + "=" * 80)
        print(f"📊 SUMMARY: Processed {len(results)}/{len(all_files)} files successfully")
        print("=" * 80)
        
        # 📦 Save consolidated results
        if results:
            # Save consolidated JSON with all results
            consolidated_file = output_dir / "all_directives_extracted.json"
            consolidated_dict = {result['document_id']: result for result in results}
            with open(consolidated_file, 'w', encoding='utf-8') as f:
                json.dump(consolidated_dict, f, indent=2, ensure_ascii=False)
            print(f"\n📦 Consolidated results saved to: {consolidated_file}")
            
            # Save CSV summary
            import pandas as pd
            rows = []
            for result in results:
                row = {
                    'document_id': result['document_id'],
                    'category': result['category'],
                    'confidence': result['legalbert_confidence'],
                    'status': result['processing_status'],
                    'title': result['extracted_info'].get('title', ''),
                    'effective_date': result['extracted_info'].get('effective_date', ''),
                    'affected_sectors': '; '.join(result['extracted_info'].get('affected_sectors', [])) if isinstance(result['extracted_info'].get('affected_sectors'), list) else result['extracted_info'].get('affected_sectors', ''),
                }
                rows.append(row)
            
            df = pd.DataFrame(rows)
            csv_file = output_dir / "directives_summary.csv"
            df.to_csv(csv_file, index=False)
            print(f"📊 Summary CSV saved to: {csv_file}")
        
        # Display summary of all results
        print("\n📋 DETAILED RESULTS:")
        print("-" * 80)
        for result in results:
            print(f"\n📄 {result['document_id']}")
            print(f"   Category: {result['category']}")
            print(f"   Confidence: {result['legalbert_confidence']:.2%}")
            print(f"   Status: {result['processing_status']}")
    else:
        print("⚠️ No HTML or XML files found in the directives directory")
else:
    print(f"❌ Directives directory not found: {directives_dir}")

## 4. Using the Standalone Script (Lambda-ready)

The processing logic has been extracted to `scripts/process_regulatory_document.py` which can be:
- **Run from this notebook** using the function below
- **Executed standalone** via command line
- **Deployed to AWS Lambda** to process files automatically when they're uploaded to S3

The script supports both local files and S3 paths.

In [ ]:
import sys
from pathlib import Path

# Ajouter la racine du projet au path pour permettre l'import avec scripts.
# Depuis notebooks/extraction/, remonter à la racine du projet
project_root = Path().cwd()
if 'notebooks' in str(project_root):
    # On est dans notebooks/extraction/, remonter de 2 niveaux
    project_root = project_root.parent.parent
elif 'notebooks' in str(project_root.parent):
    # On est dans notebooks/, remonter d'un niveau
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from scripts.processing.process_regulatory_document import RegulatoryDocumentProcessor as LambdaProcessor

def process_single_file_with_script(file_path: str, output_dir: str = '../../data/generated/extracted_directives'):
    """
    Process a single regulatory file using the Lambda-ready script.
    
    Args:
        file_path: Path to the regulatory file (local or s3://)
        output_dir: Directory to save the output
    
    Returns:
        Dict with processing results
    """
    from pathlib import Path
    
    # Initialize processor
    processor = LambdaProcessor(region_name='us-east-1')
    
    # Process document
    result = processor.process_document(file_path)
    
    # Generate output path
    document_name = file_path.split('/')[-1]
    file_stem = Path(document_name).stem
    output_path = Path(output_dir) / f"{file_stem}_extracted.json"
    
    # Save result
    processor.save_result(result, str(output_path))
    
    return result

# Example: Process a single file
file_to_process = '../../data/raw/directives/4.REGULATION (EU) 20241689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL.html'
if os.path.exists(file_to_process):
    print(f"🔄 Processing single file using Lambda-ready script...")
    result = process_single_file_with_script(file_to_process)
    print(f"\n✅ Processing completed!")
    print(f"   Document: {result['document_id']}")
    print(f"   Category: {result['category']}")
    print(f"   Status: {result['processing_status']}")
else:
    print(f"⚠️ File not found: {file_to_process}")